# 02 — Sensor / line-of-sight footprint

God's Eye View presents a sensor-style global view and tracks objects in 3D. This notebook explores a related geometry question: **how much of Earth's surface is geometrically visible from a satellite at a given altitude?**

The calculation is an educational spherical-Earth approximation. It does not use or reproduce upstream third-party imagery, camera feeds, or other datasets.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
R_E = 6378.137  # km

def horizon_central_angle(altitude_km):
    return np.arccos(R_E / (R_E + altitude_km))

def horizon_radius_km(altitude_km):
    return R_E * horizon_central_angle(altitude_km)

for h in [200, 400, 550, 1200, 20200, 35786]:
    ang = np.rad2deg(horizon_central_angle(h))
    print(f"{h:6.0f} km  horizon central angle={ang:6.2f}°  surface radius={horizon_radius_km(h):8.0f} km")


In [ ]:
alts = np.linspace(160, 2000, 300)
radii = horizon_radius_km(alts)
fig, ax = plt.subplots(figsize=(9,4.8))
ax.plot(alts, radii)
ax.set(xlabel="Satellite altitude (km)", ylabel="Geometric horizon radius (km)", title="Visible surface radius grows with altitude")
ax.grid(True, alpha=.3); plt.show()


In [ ]:
def footprint_ring(lat0_deg, lon0_deg, radius_km, points=361):
    lat0, lon0 = np.deg2rad([lat0_deg, lon0_deg])
    delta = radius_km / R_E
    bearing = np.linspace(0, 2*np.pi, points)
    lat = np.arcsin(np.sin(lat0)*np.cos(delta) + np.cos(lat0)*np.sin(delta)*np.cos(bearing))
    lon = lon0 + np.arctan2(np.sin(bearing)*np.sin(delta)*np.cos(lat0), np.cos(delta)-np.sin(lat0)*np.sin(lat))
    lon = (lon + np.pi) % (2*np.pi) - np.pi
    return np.rad2deg(lat), np.rad2deg(lon)

sub_lat, sub_lon = 28.5, -80.6
altitude_km = 550
ring_lat, ring_lon = footprint_ring(sub_lat, sub_lon, horizon_radius_km(altitude_km))
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ring_lon, ring_lat)
ax.scatter([sub_lon],[sub_lat],s=60,label="sub-satellite point")
ax.set(xlim=(-180,180), ylim=(-90,90), xlabel="Longitude (deg)", ylabel="Latitude (deg)", title=f"Geometric horizon footprint at {altitude_km:.0f} km")
ax.grid(True, alpha=.3); ax.legend(); plt.show()


**Interpretation:** this is a geometric line-of-sight limit, not an instrument's usable field of view. A real payload footprint also depends on sensor boresight, field-of-view angle, terrain, atmosphere, mission constraints, and resolution.
